# Imports

In [12]:
%load_ext autoreload
%autoreload 2
import time

import jax
import jax.numpy as jnp

from eci.environment import Environment
from eci.voting_system.beliefs import _get_current_beliefs, _get_pref_belief_gap
from eci.voting_system.decisions import _compute_option_preferences, _sample_choice

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# How an agent make a choice

In [ ]:
env = Environment(num_voters=10, num_candidates=3, num_preferences=2)

In [14]:
# Here add a step with explanation of pyhgf and what are doing initialise network

In [ ]:
# TODO: a way to stop sometimes the input data
env.initialize_network()

In [ ]:
all_agent_data = _get_current_beliefs(env)
all_agent_data

{0: {'means_belief': Array([2.5094445, 2.2970998], dtype=float32),
  'precisions_belief': Array([3.5426784, 3.7848213], dtype=float32),
  'means_preference': Array([1.008174, 1.008174], dtype=float32),
  'precision_preference': Array([0.6436357, 0.6436357], dtype=float32),
  'mean_policy': Array([[1.308671  , 0.06526518],
         [1.9061606 , 0.4126644 ],
         [0.98256636, 0.6757083 ]], dtype=float32),
  'precision_policy': Array([[0.78398144, 0.94756657],
         [0.4873248 , 0.82047296],
         [0.31618005, 0.34260136]], dtype=float32)},
 1: {'means_belief': Array([2.7950838, 2.2861986], dtype=float32),
  'precisions_belief': Array([0.5012235 , 0.58261603], dtype=float32),
  'means_preference': Array([0.17660117, 0.17660117], dtype=float32),
  'precision_preference': Array([0.75527155, 0.75527155], dtype=float32),
  'mean_policy': Array([[1.308671  , 0.06526518],
         [1.9061606 , 0.4126644 ],
         [0.98256636, 0.6757083 ]], dtype=float32),
  'precision_policy': Array

In [ ]:
pref_belief_gap = _get_pref_belief_gap(all_agent_data)
pref_belief_gap

Array([2.174395 , 4.3366857, 0.5993   , 1.1153344, 1.7594877, 2.9185555,
       0.54668  , 5.4194136, 0.624352 , 2.6399002], dtype=float32)

In [18]:
# maybe put this into the function
means_belief = jnp.stack(
    [agent_data["means_belief"] for agent_data in all_agent_data.values()]
)
precisions_belief = jnp.stack(
    [agent_data["precisions_belief"] for agent_data in all_agent_data.values()]
)

In [ ]:
# TODO: make a manual exemple to be sure
candidate_preferences = _compute_option_preferences(
    env,
    means_belief,
    precisions_belief,
    pref_belief_gap,
)
candidate_preferences

Array([[-1.4331374 , -0.3045349 , -0.1436441 ],
       [ 1.005238  ,  2.6710157 ,  3.2677474 ],
       [-2.7053094 , -1.2618879 , -0.85972226],
       [-2.2395005 , -0.82520497, -0.4504646 ],
       [-1.6732954 , -0.3045386 ,  0.02990139],
       [-0.36038613,  1.1018052 ,  1.5216578 ],
       [-2.797998  , -1.3781264 , -0.99818945],
       [ 2.1729455 ,  3.725517  ,  4.226035  ],
       [-2.6952019 , -1.2610304 , -0.8676244 ],
       [-0.6739683 ,  0.7636293 ,  1.1602794 ]], dtype=float32)

In [20]:
key = jax.random.PRNGKey(int(time.time()))
key_round_1, key_round_2 = jax.random.split(key)

In [21]:
mask_round_1 = jnp.ones_like(candidate_preferences, dtype=bool)
masked_preferences = jnp.where(mask_round_1, candidate_preferences, -jnp.inf)

In [22]:
# Sample round 1 vote
vote_1, softmax_probs_1 = _sample_choice(key_round_1, masked_preferences)
vote_1

Array([1, 1, 2, 2, 1, 2, 2, 2, 0, 0], dtype=int32)